# 1. Import Libraries

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount('/content/drive')

# 2. Set Datapaths, Image Class, and Height & Width

In [ ]:
# 6 class names
CLASS_NAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
NUM_CLASSES = 6
IMG_HEIGHT = 64
IMG_WIDTH = 64
IMG_SIZE = (IMG_WIDTH, IMG_HEIGHT)
RANDOM_SEED = 42

# Predicition datapath
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/seg_pred/seg_pred/"

# Training and Test image datapaths
TRAIN_DIR = "/content/drive/MyDrive/Colab Notebooks/seg_train/seg_train/"
TEST_DIR  = "/content/drive/MyDrive/Colab Notebooks/seg_test/seg_test/"

# 3. Count Images

In [ ]:
total_train = 0

# Loop through TRAIN_DIR folder
for class_name in CLASS_NAMES:
    class_path = TRAIN_DIR + class_name
    file_count = len(os.listdir(class_path))
    total_train = total_train + file_count
    print(class_name, "->", file_count, "images")

print("─" * 35)
print("Total training images:", total_train)

total_test = 0
# Loop through TEST_DIR folder
for class_name in CLASS_NAMES:
    class_path = TEST_DIR + class_name
    file_count = len(os.listdir(class_path))
    total_test = total_test + file_count
    print(class_name, "->", file_count, "images")

print ("-" * 35)
print ("Total testing images", total_test)

# 4. Format into RGB and Apply Labels

In [ ]:
def load_images(folder_path, class_names, img_size):
    images = []
    labels = []

    # Loop through each class
    for index, class_name in enumerate(class_names):
        class_path = folder_path + class_name

        print("Loading:", class_path)

        # Looping through every image file in this class folder
        for filename in os.listdir(class_path):
            if not filename.endswith(('.jpg', '.jpeg', '.png')):
                continue                               # skip, go back to top of loop

            image_path = class_path + "/" + filename
            img = cv2.imread(image_path) # Read images

            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert cvtColor from standard BGR to RGB
            img = cv2.resize(img, img_size)
            images.append(img)
            labels.append(index)

    return np.array(images), np.array(labels)

# Call function
X_train_full, y_train_full = load_images(TRAIN_DIR, CLASS_NAMES, IMG_SIZE)
X_test, y_test = load_images(TEST_DIR,  CLASS_NAMES, IMG_SIZE)

print( "Training images shape:", X_train_full.shape)
print("Test images shape:",     X_test.shape)

# 5. Split Data into Train, Validate, and Test

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size = 0.15, # Spliting by 85% and 15%
    stratify = y_train_full, # Maintain distribution so classes don't overpower one anothers
    random_state = RANDOM_SEED
)

print("Training set:  ", X_train.shape[0], "images")
print("Validation set:", X_val.shape[0],   "images")
print("Test set:      ", X_test.shape[0],  "images")

# 6. Normalize Images to convert to Probability

In [ ]:
X_train = X_train.astype('float32') / 255.0 # Manually Normalizing each image from float32 to 0-1 for Training, Validating, Testing
X_val = X_val.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print("Pixel range after normalization:", X_train.min(), "to", X_train.max())

# 7. Display Images with labels

In [ ]:
# Using matplotlib to show a 15 row, 6 column grid
plt.figure(figsize=(15, 6))

for i in range(NUM_CLASSES):

    # Index of the first image that belongs to class 'i' which should return the indices where condition is true
    first_index = np.where(y_train == i)[0][0]

    # Create subplot position
    plt.subplot(2, NUM_CLASSES, i + 1)

    plt.imshow(X_train[first_index])

    # Next we label it with the class name
    plt.title(CLASS_NAMES[i])
    plt.axis('off')   # hides the x/y axis nums

plt.suptitle("One sample image per class", fontsize=14)
plt.tight_layout()
plt.show()

# 8. Save Data

In [ ]:
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_val.npy",   X_val)
np.save("y_val.npy",   y_val)
np.save("X_test.npy",  X_test)
np.save("y_test.npy",  y_test)

print("Total images processed:", X_train.shape[0] + X_val.shape[0] + X_test.shape[0])